# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)

# Display summary info
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields and columns. All references are given by their `@id`.

In [ ]:
# Enumerate all record sets with their @id, name, and available fields/columns

def show_recordsets(ds):
    print("Available Record Sets:")
    for record_set in ds.record_sets:
        print(f"- RecordSet @id: {record_set.id}")
        print(f"  name: {getattr(record_set, 'name', None)}")
        print(f"  description: {getattr(record_set, 'description', None)}")
        field_ids = [f.id for f in record_set.fields] if getattr(record_set, 'fields', None) else []
        print(f"  Field @ids: {field_ids}")
        for field in getattr(record_set, 'fields', []):
            print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
        if hasattr(record_set, 'columns'):
            column_ids = [c.id for c in record_set.columns]
            print(f"  Column @ids: {column_ids}")
            for column in record_set.columns:
                print(f"    - Column @id: {column.id}, name: {getattr(column, 'name', None)}, dataType: {getattr(column, 'data_type', None)}")
        print()

show_recordsets(dataset)

## 3. Data Extraction
Load data for a selected record set into a DataFrame for analysis. Use `@id` values from the overview above.

Below we demonstrate extracting the main protein abundance record set.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record Set @ids found: {record_set_ids}")

# Example: Extract the main record set (typically the first one, update if necessary)
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"Main record set to use: {main_record_set_id}")

dataframes = {}
if main_record_set_id is not None:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Fields (columns) in the DataFrame for record set {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head(5))
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps such as filtering, normalization, and grouping on the data. All columns are referenced by their Croissant Field or Column `@id`.

In [ ]:
# Example analysis: filter by peptide count (assume the field @id 'cr:peptideCount') if available
# Update 'cr:peptideCount' to one of the available numeric field @ids listed in the overview above if different
numeric_field_id = None
group_field_id = None
if main_record_set_id is not None:
    # Attempt to pick a reasonable numeric field, searching the record set's fields
    for rs in dataset.record_sets:
        if rs.id == main_record_set_id:
            for f in getattr(rs, 'fields', []):
                if getattr(f, 'data_type', None) in ['Float', 'Integer', 'Number']:
                    numeric_field_id = f.id
                    break
            # Pick a group field that's likely categorical
            for f in getattr(rs, 'fields', []):
                if getattr(f, 'data_type', None) in ['Text', 'String']:
                    group_field_id = f.id
                    break

    if numeric_field_id is None:
        print("Couldn't auto-detect a numeric field. Please set 'numeric_field_id' to an available numeric field @id.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")

    # Use the DataFrame
    df = dataframes[main_record_set_id]

    # Check the field exists
    if numeric_field_id in df.columns:
        # Set threshold, e.g., 10 for demonstration
        threshold = 10

        # Drop rows with missing values in the numeric column
        filtered_df = df[df[numeric_field_id].notnull()]
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (first 5):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a text/categorical field if one exists
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_field_id} (first 5):")
            display(grouped_df.head())
        else:
            print("No categorical field was found for grouping.")
    else:
        print(f"Field {numeric_field_id} not found in columns. Columns are: {df.columns.tolist()}")
else:
    print("No record sets or columns available to analyze.")

## 5. Visualization
Visualize the distribution of the filtered numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]

    # Prepare filtered data as in previous section
    filtered_df = df[df[numeric_field_id].notnull()]
    filtered_df = filtered_df[filtered_df[numeric_field_id] > 10]

    # Plot the distribution
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20, color='cornflowerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")

    plt.subplot(1,2,2)
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col not in filtered_df:
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    sns.histplot(filtered_df[norm_col], kde=True, bins=20, color='orange')
    plt.title(f"Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot: No suitable numeric field detected.")

## 6. Conclusion
In this notebook, we've explored the FAIR^2 dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" via its Croissant schema. We've identified the available record sets and fields by their unique `@id`, loaded the primary data into a DataFrame, and performed basic filtering, normalization, grouping, and visualization. These steps illustrate how `mlcroissant` helps navigate complex, FAIR-aligned multi-table datasets programmatically.